# Day 4: Generalize, Dashboard, Report

## Findings (Days 1 to 3)
**Data**
- 1,187,827 requests for humblebee.ai and hbai.dev, 2026-10-21 14:38 to 2026-10-29 15:13 (KST).
- The JSON log silently skips static files: 32,299 requests exist only in the text log. Likely a `location` block with its own `access_log`.
- `$remote_user` carried Langfuse API keys and an admin email. Dropped.

**Traffic**
- Machines dominate: monitoring 35.3%, browsers 31.0%, scripts 21.8%, AI agents 7.7%.
- The honeypot takes 328,713 requests (90.8% crawlers) and 96.5% of all secret-file probes.
- Location comes from Cloudflare headers; requests that skip Cloudflare had none. Filled from the same IP; 28,039 rows still unknown.

**Errors**
- Down all week: speakology (11,874 x 502), smart-office, repo-holmes (GitHub webhooks lost).
- Shared outages on Oct 28 13:35 and 17:35: up to 12 services failing in the same 10 minutes.
- hstaff `/mcp`: 352 nginx errors, all to one upstream host. Refused connections and read timeouts on long MCP streams.
- A Beszel agent retries with a bad key: 19,554 x 401.
- Reloads do not cause errors. No evidence of leaked secret files: 200s on secret paths are catch-all pages.

## 1. Run the loader on every domain
`loader.py` holds the Day 1 functions unchanged. If it reproduces Day 1 exactly, it can be trusted on the other domains.

In [1]:
import pandas as pd
import plotly.express as px
import loader

day1 = pd.read_parquet('data/logs.parquet')
again = pd.concat([loader.load_domain(d) for d in ['humblebee.ai', 'hbai.dev']], ignore_index=True)
len(again) == len(day1), set(again.request_id) == set(day1.request_id)

(True, True)

In [2]:
frames = {d: loader.load_domain(d) for d in loader.domains()}
summary = pd.DataFrame({
    d: {'requests': len(x),
        'subdomains': x.subdomain.nunique() if len(x) else 0,
        'text_gap': (x.source == 'text_gap').sum() if len(x) else 0}
    for d, x in frames.items()
}).T.sort_values('requests', ascending=False)
summary

,requests,subdomains,text_gap
humblebee.ai,858999,37,17564
hbai.dev,328828,22,14735
lumiohub.ai,120509,4,5313
kumush.ai,50091,2,1510
beeintel.ai,20807,1,638
nyaie.org,9481,1,908
azam-electric.beeintel.ai,4555,1,1721
budget.humblebee.ai,679,1,138
learnprompting.uz,325,1,92
cip.lumiohub.ai,4,1,0


Domains with 0 requests only have empty files or `.gitkeep`. No file in any domain failed to parse: `load_text` would have printed bad lines.
The JSON gap (`text_gap`) appears in every domain that has JSON logs, so it is a server-wide nginx config issue, not one site.

In [3]:
all_logs = pd.concat([x for x in frames.values() if len(x)], ignore_index=True)
len(all_logs), all_logs.domain.nunique()

(1394278, 10)

## 2. Enrich once, for the dashboard
Same IP-kind and location fill as Day 2, applied to all domains.

In [4]:
import ipaddress

CGNAT = ipaddress.ip_network('100.64.0.0/10')

def ip_kind(ip):
    try:
        a = ipaddress.ip_address(ip)
    except ValueError:
        return 'invalid'
    return 'internal' if a.is_private or a in CGNAT else 'public'

all_logs['ip_kind'] = all_logs.client_address.map({ip: ip_kind(ip) for ip in all_logs.client_address.unique()})

for col in ['country', 'city']:
    known = all_logs.dropna(subset=[col]).groupby('client_address')[col].agg(lambda s: s.mode()[0])
    all_logs[col] = all_logs[col].fillna(all_logs.client_address.map(known))
    all_logs.loc[all_logs.ip_kind == 'internal', col] = 'internal'
    all_logs[col] = all_logs[col].fillna('unknown')

all_logs.country.value_counts().head(8)

country
KR          569727
US          193045
UZ          150237
internal    147652
DE          100859
IN           69207
NL           51209
unknown      31885
Name: count, dtype: int64

In [5]:
all_logs.to_parquet('data/logs_all.parquet')
check = pd.read_parquet('data/logs_all.parquet')
len(check) == len(all_logs)

True

## 3. Status per domain

In [6]:
all_logs['status_class'] = (all_logs.status_code // 100).astype(str) + 'xx'
pd.crosstab(all_logs.domain, all_logs.status_class, normalize='index').round(3).assign(
    requests=all_logs.domain.value_counts()).sort_values('requests', ascending=False)

status_class,1xx,2xx,3xx,4xx,5xx,requests
domain,,,,,,
humblebee.ai,0.003,0.531,0.009,0.436,0.022,858999
hbai.dev,0.013,0.794,0.059,0.129,0.004,328828
lumiohub.ai,0.006,0.939,0.025,0.028,0.002,120509
kumush.ai,0.128,0.510,0.219,0.141,0.002,50091
beeintel.ai,0.000,0.981,0.018,0.001,0.000,20807
nyaie.org,0.000,0.243,0.042,0.713,0.001,9481
azam-electric.beeintel.ai,0.000,0.886,0.096,0.016,0.002,4555
budget.humblebee.ai,0.000,0.947,0.013,0.034,0.006,679
learnprompting.uz,0.000,0.960,0.000,0.040,0.000,325


In [7]:
daily = all_logs[all_logs.subdomain != 'a-honeypot'].groupby([pd.Grouper(key='datetime', freq='1D'), 'domain']).size().reset_index(name='requests')
px.bar(daily, x='datetime', y='requests', color='domain', title='Requests per day by domain (honeypot excluded)')

## 4. Dashboard
`dashboard.py` reads `data/logs_all.parquet` (falls back to `data/logs.parquet`). Run it from this folder:

```
streamlit run dashboard.py
```

Filters: domain, subdomain, dates, status class, hide honeypot. Shows requests, 5xx share, median response time, unique IPs, requests per hour, top subdomains, top countries, slowest paths.
Query strings are cut before display, so tokens in URLs never appear.

### Streamlit in two ideas
- **Rerun model**: every click reruns the whole script top to bottom. Widgets return their current value, so the code reads like a plain script.
- **`@st.cache_data`**: the Parquet load runs once and is reused on every rerun. Without it, each click would reload 1.4 million rows.